# Wikibase import pipeline

This notebook imports the core entities of the EFF knowledge graph into Wikibase.

It can create or update several types of resources: properties, base taxonomy items, ports, port statistics, and vessel entities. The import is controlled through `RUN_*` flags, so each block can be executed independently depending on the current stage of the pipeline.

The vessel import is the main operational section. It reads consolidated vessel CSV files, resolves existing entities by CFR or external code, builds Wikibase statements with references, creates missing ports when needed, and writes the resulting data to the target Wikibase instance.

Before running this notebook, make sure that:
- the `.env` file contains valid `WB_USER` and `WB_PASS` credentials.
- the expected CSV files exist in the configured Drive folder.
- the target Wikibase property and item mappings match the constants defined below.
- `INPUT_MODE` is set correctly (`eu_only` or `national_eu`).
- only the intended `RUN_*` flags are enabled.

# Initial setup

This section mounts Google Drive, loads credentials from the `.env` file, installs required dependencies, and initializes the Wikibase login session.

It also defines the target Wikibase endpoint, global execution mode, and the `RUN_*` switches that control which parts of the notebook will execute.

Only enable one import stage at a time unless you explicitly want to run several stages in sequence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/BCKG/CSV Barcos"

from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/BCKG/.env')

In [ ]:
!pip install wikidataintegrator --quiet
!pip install lightrdf --quiet

In [ ]:
import wikidataintegrator as WI
from datetime import datetime
import csv
import glob
import json
import logging
import os
import re
import sys
import time

import requests

bot_user=os.getenv("WB_USER")
bot_pwd=os.getenv("WB_PASS")

if not bot_user or not bot_pwd:
    raise ValueError("Missing WB_USER or WB_PASS environment variables")

wikibase_url = "https://effkg.wikibase.cloud"
mediawiki_api_url = "https://effkg.wikibase.cloud/w/api.php"
login_instance = WI.wdi_login.WDLogin(user=bot_user, pwd=bot_pwd, mediawiki_api_url=mediawiki_api_url)
WI.wdi_config.config['REFETCH'] = False
SESSION = requests.Session()

INPUT_MODE = "eu_only"   # or "national_eu"

RUN_PROPERTIES = False
RUN_BASE_ITEMS = False
RUN_PORTS = False
RUN_PORT_STATS = False
RUN_VESSELS = True

In [ ]:
import logging

logging.basicConfig(
    filename="wikibaseintegration.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

log = logging.getLogger()

# Properties

This optional section creates Wikibase properties from `add_properties.csv`.

Each row defines a property label, datatype, description, and Spanish label. This block should normally be run only during the initial setup of a new Wikibase instance.

Do not run this section again unless you intentionally want to create additional properties.

In [ ]:
if RUN_PROPERTIES:
  csv_path = './add_properties.csv'
  log.debug(f'Using file: {csv_path}')

  with open(csv_path, 'r', encoding='latin-1') as csv_file:
      reader = csv.reader(csv_file, delimiter=';')
      headers = next(reader)

      for row in reader:
          property_label_en = row[0]
          datatype = row[1]
          property_description = row[2]
          property_label_es = row[3]

          property = WI.wdi_core.WDItemEngine(mediawiki_api_url=mediawiki_api_url, data=[])
          property.set_label(lang="en", label=property_label_en)
          property.set_label(lang="es", label=property_label_es)

          result = property.write(login=login_instance, entity_type='property', property_datatype=datatype)

          log.debug(f"Created property: {result}")

# Base entities

This optional section may be adapted to load required items from a csv file.

After making the necessary changes, run this section only when bootstrapping a new Wikibase instance or when adding new base vocabulary items.

In [ ]:
if RUN_BASE_ITEMS:
  csv_path = './items.csv'
  complete_data = []
  log.debug(f'Using file: {csv_path}')

  try:
      with open(csv_path, 'r', encoding='latin-1') as csv_file:
          reader = csv.reader(csv_file, delimiter=';')
          headers = next(reader)

          for row in reader:

              item_label_en = row[0].strip()
              item_label_es = row[1].strip()
              description_en = row[2].strip()

              log.debug(f"Processing item: EN='{item_label_en}', ES='{item_label_es}', Desc='{description_en}'")

              data = []

              for i in range(3, len(row)):
                  claim_value = row[i].strip()

                  if claim_value:
                      header_parts = headers[i].strip().split(':', 1)

                      if len(header_parts) < 2:
                          log.debug(f"Warning: Skipping column '{headers[i]}' for item '{item_label_en}' - header format should be 'Px:datatype'")
                          continue

                      prop_id = header_parts[0].strip()
                      datatype = header_parts[1].strip()

                      if datatype == 'wikibase-item':
                          # For wikibase-item, the value in the CSV should be a Q-id (e.g., "Q123")
                          if not claim_value.startswith('Q') or not claim_value[1:].isdigit():
                              log.debug(f"Warning: Skipping claim '{prop_id}' for item '{item_label_en}' with value '{claim_value}' - expected a QID for wikibase-item datatype.")
                              continue

                          statement = WI.wdi_core.WDItemID(value=claim_value, prop_nr=prop_id)
                          data.append(statement)

              try:
                  item = WI.wdi_core.WDItemEngine(data=data, mediawiki_api_url=mediawiki_api_url)

                  if item_label_en:
                      item.set_label(lang="en", label=item_label_en)
                  if item_label_es:
                      item.set_label(lang="es", label=item_label_es)

                  if description_en:
                      item.set_description(lang="en", description=description_en)
                  result = item.write(login=login_instance, bot_account=True)

                  if result:
                      log.debug(f"Successfully created item: {item_label_en} ({result})")
                      new_row = row.copy()
                      new_row.append(result)
                      complete_data.append(new_row)
                  else:
                      log.error(f"Error: No QID returned after writing item {item_label_en}. Result: {result}")
                      complete_data.append(row.copy() + ['NO_QID_RETURNED'])

              except Exception as write_error:
                  log.error(f"Error writing item '{item_label_en}': {write_error}")
                  complete_data.append(row.copy() + ['WRITE_ERROR'])

  except FileNotFoundError:
          log.error(f"Error: CSV file not found at {csv_path}")
          sys.exit(1)
  except ValueError as ve:
        log.error(f"Configuration Error: {ve}")
        sys.exit(1)
  except Exception as e:
      log.error(f"An unexpected error occurred during CSV processing: {e}")
      sys.exit(1)


# Utility functions

This section defines reusable helper functions used across the import pipeline.

The utilities include:
- SPARQL query execution.
- item lookup by label.
- item loading by QID.
- lookup indexes for vessel types, fishing gear, ports, and existing vessels.
- normalization helpers for names, ports, and identifiers.
- reference builders.
- dry-run comparison helpers.

Most later sections depend on these functions, so this section must be executed before ports, statistics, or vessels.

In [ ]:
"""Execute a SPARQL query against the configured Wikibase endpoint."""
def execute_sparql_query(prefix='', query='', endpoint=f'{wikibase_url}/query/sparql',
                         user_agent="User Agent Name"):
    quoute_check = re.search(r"\s'(.*)'\s.", query)
    if quoute_check is not None:
        replace_with = quoute_check[1].replace("'","\\'")
        query = query.replace(quoute_check[1],replace_with)


    params = {
        'query': prefix + '\n#Tool: PBB_core fastrun\n' + query,
        'format': 'json'
    }

    headers = {
        'Accept': 'application/sparql-results+json',
        'User-Agent': user_agent
    }
    response = SESSION.get(endpoint, params=params, headers=headers, timeout=60)
    response.raise_for_status()

    return response.json()

"""Search for an item by exact label match in a given language."""
def searchForItemUsingLabel(base_url,label,language,login_instance):
    item = None
    url = base_url+f"/w/api.php?action=wbsearchentities&search={label}&language={language}&format=json&type=item"
    response = SESSION.get(url)
    search_results = response.json()

    if search_results['search']:
        for result in search_results['search']:
                if result["match"]["text"] == label:
                    qid = result['id']
                    log.debug(f"QID encontrado: {qid}")
                    return qid
    return item

"""Shortcut for English label lookup."""
def searchEnLabel(label):
  return searchForItemUsingLabel(wikibase_url,label,"en",login_instance)

"""Load an item entity from Wikibase by QID."""
def loadEnID(qid):
  log.debug(f"Loading {qid}")
  return WI.wdi_core.WDItemEngine(wd_item_id=qid, mediawiki_api_url=mediawiki_api_url)

"""Build an in-memory mapping between vessel type abbreviations and QIDs."""
def build_vessel_type_index():
    query = f'''
        PREFIX wd: <{wikibase_url}/entity/>
        PREFIX wdt: <{wikibase_url}/prop/direct/>

        SELECT ?item ?abbr WHERE {{
          ?item wdt:P35 wd:Q399 ;
                wdt:P53 ?abbr .
        }}
    '''
    results = execute_sparql_query(query)['results']['bindings']

    index = {}
    for r in results:
        qid = r['item']['value'].split('/')[-1]
        abbr = r['abbr']['value'].strip()
        if abbr and abbr not in index:
            index[abbr] = qid

    return index

vessel_type_index = build_vessel_type_index()

"""Resolve the vessel category QID from a vessel type code."""
def get_vessel_category_qid(vessel_type):
    if vessel_type is None:
        return "Q466"
    vessel_type = vessel_type.strip()
    if INPUT_MODE == "eu_only":
      return vessel_type_index.get(vessel_type, "Q446")
    else:
      return vessel_type_index.get(vessel_type, "Q466")

"""Build an in-memory mapping between fishing gear abbreviations and QIDs."""
def build_fishing_gear_index():
    query = f'''
        PREFIX wd: <{wikibase_url}/entity/>
        PREFIX wdt: <{wikibase_url}/prop/direct/>

        SELECT ?item ?abbr WHERE {{
          ?item wdt:P35 wd:Q472 ;
                wdt:P53 ?abbr .
        }}
    '''
    results = execute_sparql_query(query)['results']['bindings']

    index = {}
    for r in results:
        qid = r['item']['value'].split('/')[-1]
        abbr = r['abbr']['value']
        if abbr and abbr not in index:
            index[abbr] = qid

    return index

fishing_gear_index = build_fishing_gear_index()

"""Resolve a fishing gear abbreviation into a Wikibase QID."""
def get_fishing_gear_qid(gear):
    if gear is None:
        return "Q546"
    return fishing_gear_index.get(gear, "Q546")

"""Attach references to a statement based on configured data origins."""
def add_new_reference(statement, column, origin_override=None, cfr=None):
    if INPUT_MODE == "eu_only":
        origins = ["eu"]
    elif origin_override is not None:
        if isinstance(origin_override, list):
            origins = origin_override[:]
        else:
            origins = [origin_override]
    else:
        origins = properties[column]["origin"][:]

    if cfr == "-":
        origins = [o for o in origins if o == "nat"]

    for origin in origins:
      if origin == "eu":
        reference = WI.wdi_core.WDUrl(
            value="https://webgate.ec.europa.eu/fleet-europa",
            prop_nr=ref_prop,
            is_reference=True
        )
        statement.references.append([reference])
        log.debug(f"Added reference {ref_prop} -> https://webgate.ec.europa.eu/fleet-europa")
      elif origin == "nat":
        url = "https://servicio.pesca.mapama.es/CENSO/ConsultaBuqueRegistro/Buques/Search"
        reference = WI.wdi_core.WDUrl(
            value=url,
            prop_nr=ref_prop,
            is_reference=True
        )
        statement.references.append([reference])
        log.debug(f"Added reference {ref_prop} -> {url}")
    return statement

EU_PORTS_SOURCE_URL = "https://msi.nga.mil"

"""Attach the EU ports source reference to a statement."""
def add_eu_port_csv_reference(statement):
    reference = WI.wdi_core.WDUrl(
        value=EU_PORTS_SOURCE_URL,
        prop_nr=ref_prop,
        is_reference=True
    )
    statement.references.append([reference])
    return statement

"""Check whether a vessel name should be treated as missing."""
def is_missing_vessel_name(value):
    if value is None:
        return True
    normalized = str(value).strip().upper()
    return normalized in {"", "NO NAME", "<NO NAME GIVEN>", "NO NAME GIVEN"}

In [ ]:
"""Normalize text values for case-insensitive matching."""
def normalize_name(value):
    if not value:
        return None

    value = value.strip().lower()

    value = re.sub(r'\s+', ' ', value)

    return value

"""Remove trailing clarification text from a port name."""
def strip_port_clarification(value):
    if not value:
        return None
    value = value.strip()
    value = re.sub(r"\s*\([^)]*\)\s*$", "", value).strip()
    return value or None

"""Generate normalized candidate variants for a port name."""
def candidate_port_names(value):
    candidates = []
    if value:
        candidates.append(value.strip())

    stripped = strip_port_clarification(value)
    if stripped and stripped not in candidates:
        candidates.append(stripped)

    normalized = []
    seen = set()
    for c in candidates:
        n = normalize_name(c)
        if n and n not in seen:
            normalized.append(n)
            seen.add(n)
    return normalized

"""Attach a row-specific source URL as a reference."""
def add_row_source_reference(statement, source_value):
    if not source_value:
        return statement
    reference = WI.wdi_core.WDUrl(
        value=source_value.strip(),
        prop_nr=ref_prop,
        is_reference=True
    )
    statement.references.append([reference])
    return statement

"""Build an index of existing ports stored in Wikibase."""
def build_port_index():
    query = f"""
    PREFIX wd: <{wikibase_url}/entity/>
    PREFIX wdt: <{wikibase_url}/prop/direct/>

    SELECT ?port ?name WHERE {{
        ?port wdt:P35 wd:Q62 ;
              wdt:P37 ?name .
    }}
    """

    results = execute_sparql_query(query)['results']['bindings']

    index = {}

    for r in results:
        qid = r['port']['value'].split('/')[-1]
        code = normalize_name(r['name']['value'])
        index[code] = qid

    return index

import glob

EU_PORTS_CSV = "../CSV Ports/wpi_european_ports.csv"
CNSP_PORTS_CSV = "../CSV Ports/cnsp_ports_ue_filtered.csv"
VESSELS_GLOB = f"./consolidated_data_by_country/{INPUT_MODE}/consolidated_data_*.csv"

"""Load and index EU ports CSV rows by normalized port name."""
def build_eu_ports_index(csv_path=EU_PORTS_CSV):
    index = {}

    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            for name in candidate_port_names(row.get("Main Port Name")):
                if name not in index:
                    index[name] = row

    return index

"""Load and index CNSP ports CSV rows by normalized port name."""
def build_cnsp_ports_index(csv_path=CNSP_PORTS_CSV):
    index = {}

    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            for name in candidate_port_names(row.get("port_name")):
                if name not in index:
                    index[name] = row

    return index

"""Normalize identifier fields and discard invalid placeholder values."""
def clean_id(v):
    if v is None:
        return None
    v = str(v).strip()
    if v in {"", "-", "nan", "None"}:
        return None
    return v

"""Build a lookup index for existing vessels using CFR or external code."""
def build_vessel_identity_index():
    query = f"""
    PREFIX wd: <{wikibase_url}/entity/>
    PREFIX wdt: <{wikibase_url}/prop/direct/>

    SELECT ?item ?cfr ?code WHERE {{
      ?item wdt:P35 ?type .
      OPTIONAL {{ ?item wdt:P12 ?cfr . }}
      OPTIONAL {{ ?item wdt:P26 ?code . }}
    }}
    """
    results = execute_sparql_query(query=query)['results']['bindings']

    index = {}

    for r in results:
        qid = r["item"]["value"].split("/")[-1]

        cfr = clean_id(r.get("cfr", {}).get("value"))
        code = clean_id(r.get("code", {}).get("value"))

        if cfr:
            index[f"CFR::{cfr}"] = qid
        elif code:
            index[f"CODE::{code}"] = qid

    return index

"""Generate the canonical identity key for a CSV vessel row."""
def get_row_identity(row):
    cfr = clean_id(row.get("CFR"))
    code = clean_id(row.get("Code"))

    if cfr:
        return f"CFR::{cfr}"
    if code:
        return f"CODE::{code}"
    return None

"""Generate a comparable signature for a statement object."""
def statement_signature(stmt):
    try:
        prop = stmt.get_prop_nr()
    except Exception:
        prop = None

    try:
        value = stmt.get_value()
    except Exception:
        value = None

    if isinstance(value, list):
        value = tuple(value)

    return (prop, str(value))

"""Group statement signatures by property for comparison purposes."""
def group_statement_signatures(statements):
    grouped = {}
    for stmt in statements:
        sig = statement_signature(stmt)
        prop = sig[0]
        if prop not in grouped:
            grouped[prop] = set()
        grouped[prop].add(sig[1])
    return grouped

"""Load and normalize existing statements from a Wikibase item."""
def load_existing_statement_signatures(qid):
    item = loadEnID(qid)
    grouped = {}
    for stmt in item.statements:
        try:
            prop = stmt.get_prop_nr()
            value = stmt.get_value()
            if isinstance(value, list):
                value = tuple(value)
            if prop not in grouped:
                grouped[prop] = set()
            grouped[prop].add(str(value))
        except Exception:
            continue
    return grouped

dry_run_stats = {
    "processed": 0,
    "would_create": 0,
    "would_update": 0,
    "would_skip_unchanged": 0,
    "errors": 0,
    "details": []
}

In [ ]:
"""Parse coordinate strings into latitude, longitude and precision."""
def parse_position(value):
    if not value:
        return None

    parts = [p.strip() for p in value.split(",")]
    if len(parts) < 2:
        return None

    lat = float(parts[0])
    lon = float(parts[1])
    precision = float(parts[2]) if len(parts) > 2 and parts[2] else 0.0001
    return lat, lon, precision

"""Extract coordinates from a CNSP CSV row."""
def parse_cnsp_position(row):
    coordinates = (row.get("coordinates") or "").strip()
    if coordinates:
        parsed = parse_position(coordinates)
        if parsed:
            return parsed

    lat = (row.get("latitude") or "").strip()
    lon = (row.get("longitude") or "").strip()
    if lat and lon:
        return float(lat.replace(",", ".")), float(lon.replace(",", ".")), 0.0001

    return None

"""Find an existing port or create a new one from external CSV datasets."""
def find_or_create_port_from_csvs(registration_place, eu_ports_index, cnsp_ports_index):
    for normalized_port in candidate_port_names(registration_place):
        port_row = eu_ports_index.get(normalized_port)
        if port_row:
            main_port_name = (port_row.get("Main Port Name") or "").strip()
            harbor_type = (port_row.get("Harbor Type") or "").strip()
            region_name = (port_row.get("Region Name") or "").strip()
            unlocode = (port_row.get("UN/LOCODE") or "").strip()
            position = (port_row.get("Position") or "").strip()

            if not main_port_name:
                return None

            label_en = f"Port of {main_port_name}"
            label_es = f"Puerto de {main_port_name}"

            if harbor_type and region_name:
                description_en = f"{harbor_type} port in {region_name}"
                description_es = f"Puerto de tipo {harbor_type} en {region_name}"
            elif region_name:
                description_en = f"Port in {region_name}"
                description_es = f"Puerto en {region_name}"
            else:
                description_en = "Port"
                description_es = "Puerto"

            statements = []

            stmt = WI.wdi_core.WDItemID(value="Q62", prop_nr="P35")
            stmt = add_eu_port_csv_reference(stmt)
            statements.append(stmt)

            stmt = WI.wdi_core.WDString(value=main_port_name, prop_nr="P37")
            stmt = add_eu_port_csv_reference(stmt)
            statements.append(stmt)

            if unlocode:
                stmt = WI.wdi_core.WDString(value=unlocode, prop_nr="P26")
                stmt = add_eu_port_csv_reference(stmt)
                statements.append(stmt)

            parsed = parse_position(position)
            if parsed:
                lat, lon, precision = parsed
                stmt = WI.wdi_core.WDGlobeCoordinate(
                    latitude=lat,
                    longitude=lon,
                    precision=precision,
                    prop_nr="P42"
                )
                stmt = add_eu_port_csv_reference(stmt)
                statements.append(stmt)

            existing_qid = searchEnLabel(label_en) or searchEnLabel(label_es)

            if existing_qid:
                item = WI.wdi_core.WDItemEngine(
                    wd_item_id=existing_qid,
                    data=statements,
                    mediawiki_api_url=mediawiki_api_url
                )
            else:
                item = WI.wdi_core.WDItemEngine(
                    data=statements,
                    mediawiki_api_url=mediawiki_api_url
                )

            item.set_label(label_en, lang="en")
            item.set_label(label_es, lang="es")
            item.set_description(description_en, lang="en")
            item.set_description(description_es, lang="es")

            port_qid = item.write(login_instance, bot_account=True)

            port_index[normalize_name(main_port_name)] = port_qid
            stripped = strip_port_clarification(main_port_name)
            if stripped:
                port_index[normalize_name(stripped)] = port_qid

            return port_qid

    for normalized_port in candidate_port_names(registration_place):
        port_row = cnsp_ports_index.get(normalized_port)
        if port_row:
            port_name = (port_row.get("port_name") or "").strip()
            locode = (port_row.get("locode") or "").strip()
            source_value = (port_row.get("source") or "").strip()

            if not port_name:
                return None

            label_en = f"Port of {port_name}"
            label_es = f"Puerto de {port_name}"
            description_en = "Port"
            description_es = "Puerto"

            statements = []

            stmt = WI.wdi_core.WDItemID(value="Q62", prop_nr="P35")
            stmt = add_row_source_reference(stmt, source_value)
            statements.append(stmt)

            stmt = WI.wdi_core.WDString(value=port_name, prop_nr="P37")
            stmt = add_row_source_reference(stmt, source_value)
            statements.append(stmt)

            if locode:
                stmt = WI.wdi_core.WDString(value=locode, prop_nr="P26")
                stmt = add_row_source_reference(stmt, source_value)
                statements.append(stmt)

            parsed = parse_cnsp_position(port_row)
            if parsed:
                lat, lon, precision = parsed
                stmt = WI.wdi_core.WDGlobeCoordinate(
                    latitude=lat,
                    longitude=lon,
                    precision=precision,
                    prop_nr="P42"
                )
                stmt = add_row_source_reference(stmt, source_value)
                statements.append(stmt)

            existing_qid = searchEnLabel(label_en) or searchEnLabel(label_es)

            if existing_qid:
                item = WI.wdi_core.WDItemEngine(
                    wd_item_id=existing_qid,
                    data=statements,
                    mediawiki_api_url=mediawiki_api_url
                )
            else:
                item = WI.wdi_core.WDItemEngine(
                    data=statements,
                    mediawiki_api_url=mediawiki_api_url
                )

            item.set_label(label_en, lang="en")
            item.set_label(label_es, lang="es")
            item.set_description(description_en, lang="en")
            item.set_description(description_es, lang="es")

            port_qid = item.write(login_instance, bot_account=True)

            port_index[normalize_name(port_name)] = port_qid
            stripped = strip_port_clarification(port_name)
            if stripped:
                port_index[normalize_name(stripped)] = port_qid

            return port_qid

    return None

# Ports

This optional section imports or updates port entities from `puertos.csv`.

It reads port labels and statements from the CSV, searches for existing port items by label, creates or updates the corresponding Wikibase entity, and optionally links ports back to their administrative district.

Run this section before vessel import if vessels reference ports that must already exist in Wikibase.

In [ ]:
csv_path = './puertos.csv'
base_url = "https://effkg.wikibase.cloud/"
ref_prop = 'P46' #'Source'

In [ ]:
"""Attach the appropriate source reference for a port statement."""
def add_port_reference(statement, prop):
    if prop in ["P26", "P60"]:
        reference = WI.wdi_core.WDUrl(
            value="https://signa.ign.es/signa/",
            prop_nr=ref_prop,
            is_reference=True
        )
        statement.references.append([reference])
        log.debug(f"Added reference {ref_prop} -> https://signa.ign.es/signa/")

    elif prop == "P3":
        url = "https://boe.es/buscar/pdf/2007/BOE-A-2007-10951-consolidado.pdf"
        reference = WI.wdi_core.WDUrl(
            value=url,
            prop_nr=ref_prop,
            is_reference=True
        )
        statement.references.append([reference])
        log.debug(f"Added reference {ref_prop} -> {url}")

    elif prop == "P37":
        url = "https://servicio.pesca.mapama.es/CENSO/ConsultaBuqueRegistro/Buques/Search"
        reference = WI.wdi_core.WDUrl(
            value=url,
            prop_nr=ref_prop,
            is_reference=True
        )
        statement.references.append([reference])
        log.debug(f"Added reference {ref_prop} -> {url}")

    return statement

In [ ]:
"""Ensure that a district item links back to a port item."""
def ensure_port_in_district(district_qid, port_qid):

    try:
        district = WI.wdi_core.WDItemEngine(
            wd_item_id=district_qid,
            mediawiki_api_url=mediawiki_api_url
        )

        existing_claims = district.get_wd_json_representation()

        already_present = False

        if "claims" in existing_claims and "P7" in existing_claims["claims"]:
            for claim in existing_claims["claims"]["P7"]:
                target = claim["mainsnak"]["datavalue"]["value"]["id"]
                if target == port_qid:
                    already_present = True
                    break

        if already_present:
            log.debug(f"District {district_qid} already linked to port {port_qid}")
            return

        log.debug(f"Adding port {port_qid} to district {district_qid}")

        statement = WI.wdi_core.WDItemID(
            value=port_qid,
            prop_nr="P7"
        )

        district.update(data=[statement])
        district.write(login_instance, bot_account=True)

    except Exception as e:
        log.error(f"Error updating district {district_qid}: {e}")

"""
Split multiple names from the legacy CSV: 'Puerto de la Selva,,Port de la Selva'
"""
def split_port_names(value):
    return [v.strip() for v in re.split(r",,+", value) if v.strip()]

In [ ]:
import csv
import requests
import urllib.parse
import wikidataintegrator as WI

"""Find an existing item by exact English or Spanish label."""
def find_existing_item(label_en=None, label_es=None):
    if label_en:
        qid = searchForItemUsingLabel(wikibase_url, label_en, "en", login_instance)
        if qid:
            return qid

    if label_es:
        qid = searchForItemUsingLabel(wikibase_url, label_es, "es", login_instance)
        if qid:
            return qid

    return None

if RUN_PORTS:
  log.debug(f'Using file: {csv_path}')
  with open(csv_path, 'r', encoding='utf-8', newline='') as csv_file:
      reader = csv.reader(csv_file, delimiter=';')
      headers = next(reader)

      for row in reader:
          statements = []   # importante: reset por fila
          district_qid = None

          label_en = row[0].strip()
          label_es = row[1].strip()

          for i in range(2, len(row)):
              header = headers[i].strip()
              value = row[i].strip()

              # saltar vacíos
              if not value:
                  continue

              try:
                  prop, tipo = header.split(":", 1)
              except ValueError:
                  log.error(f"Header inválido: {header}")
                  continue

              try:
                  if tipo == "string":
                      if prop == "P37":
                          # dividir valores múltiples
                          values = split_port_names(value)

                          for v in values:
                              statement = WI.wdi_core.WDString(value=v, prop_nr=prop)
                              log.debug(f"  Added property {prop} with value '{v}'")
                              statement = add_port_reference(statement, prop)
                              statements.append(statement)

                      else:
                          statement = WI.wdi_core.WDString(value=value, prop_nr=prop)
                          log.debug(f"  Added property {prop} with value '{value}'")
                          statement = add_port_reference(statement, prop)
                          statements.append(statement)

                  elif tipo == "wikibase-item":
                      statement = WI.wdi_core.WDItemID(value=value, prop_nr=prop)
                      statement = add_port_reference(statement, prop)
                      statements.append(statement)
                      log.debug(f"  Added property {prop} with value '{value}'")

                      if prop == "P3":   # belongs to adm unit
                        district_qid = value

                  elif tipo == "globe-coordinate":
                      lat, lon, precision = [x.strip() for x in value.split(",")]
                      statement = WI.wdi_core.WDGlobeCoordinate(
                          latitude=float(lat),
                          longitude=float(lon),
                          precision=float(precision),
                          prop_nr=prop
                      )
                      statement = add_port_reference(statement, prop)
                      statements.append(statement)
                      log.debug(f"  Added property {prop} with value '{lat},{lon}'")

                  else:
                      log.error(f"Tipo no soportado: {tipo} en propiedad {prop}")
                      continue

              except Exception as e:
                  log.error(f"Error procesando {prop}:{tipo}='{value}' para '{label_en}': {e}")

          try:
              # 1) Buscar entidad existente
              existing_qid = find_existing_item(label_en=label_en, label_es=label_es)

              if existing_qid:
                  log.debug(f"Updating existing item {existing_qid} ({label_en})")

                  item = WI.wdi_core.WDItemEngine(
                      wd_item_id=existing_qid,
                      data=statements,
                      mediawiki_api_url=mediawiki_api_url
                  )
              else:
                  log.debug(f"Creating new item for {label_en}")

                  item = WI.wdi_core.WDItemEngine(
                      data=statements,
                      mediawiki_api_url=mediawiki_api_url
                  )

              item.set_label(label_en, lang='en')
              item.set_label(label_es, lang='es')

              result = item.write(login_instance, bot_account=True)
              log.debug(f"Saved {label_en} as {result}")

              port_qid = result

              if district_qid:
                  ensure_port_in_district(district_qid, port_qid)

          except Exception as e:
              log.error(f"Error writing item '{label_en}': {e}")


## Port statistics

This optional section imports yearly vessel-count statistics by base port.

It reads `boats_year_port.csv`, matches each port name to an existing port item, removes previous yearly count statements for that port, and writes fresh count statements with a year qualifier.

This block is destructive for the targeted statistic property: it intentionally replaces old yearly count statements before writing the new series.

In [ ]:

csv_path = './boats_year_port.csv'
ref_prop = 'P46'
stats_source_url = "https://www.mapa.gob.es/es/pesca/temas/registro-flota/informacion-sobre-flota-pesquera"

"""Build a unique index of Q62 ports for statistics updates."""
def build_q62_port_index():
    query = f"""
    PREFIX wd: <{wikibase_url}/entity/>
    PREFIX wdt: <{wikibase_url}/prop/direct/>

    SELECT ?port ?name WHERE {{
      ?port wdt:P35 wd:Q62 ;
            wdt:P37 ?name .
    }}
    """

    results = execute_sparql_query(query=query)['results']['bindings']

    index = {}
    ambiguous = set()

    for r in results:
        qid = r["port"]["value"].split("/")[-1]
        name = normalize_name(r["name"]["value"])

        if name in index and index[name] != qid:
            ambiguous.add(name)
        else:
            index[name] = qid

    # Remove ambiguous names from the index to avoid writing to the wrong port.
    for name in ambiguous:
        del index[name]

    if ambiguous:
        log.debug("Ambiguous port names skipped:")
        for name in sorted(ambiguous):
            log.debug(f"  {name}")

    return index

"""Attach the official statistics source reference to a statement."""
def add_port_stats_reference(statement):
    reference = WI.wdi_core.WDUrl(
        value=stats_source_url,
        prop_nr=ref_prop,
        is_reference=True
    )
    statement.references.append([reference])
    return statement

"""Remove existing yearly statistics statements from a port item."""
def delete_existing_p45_statements(port_qid):
    wd_item = loadEnID(port_qid)
    if not wd_item:
        log.debug(f"Could not load {port_qid}")
        return

    to_delete = [stmt.get_id() for stmt in wd_item.statements if stmt.prop_nr == "P45"]

    for stmt_id in to_delete:
        wd_item = loadEnID(port_qid)  # refresh lastrevid
        WI.wdi_core.WDItemEngine.delete_statement(
            stmt_id,
            wd_item.lastrevid,
            login_instance,
            mediawiki_api_url=mediawiki_api_url
        )

"""Build a yearly vessel count statement with year qualifier."""
def build_p45_statement(num_barcos, year):
    # P45: number of vessels (quantity).
    statement = WI.wdi_core.WDQuantity(
        value=str(num_barcos),
        prop_nr="P45",
        unit="1"
    )

    # P44: year qualifier.
    qualifier = WI.wdi_core.WDTime(
        time=f"+{int(year):04d}-01-01T00:00:00Z",
        prop_nr="P44",
        precision=9,   # year precision
        timezone=0,
        is_qualifier=True
    )
    statement.qualifiers.append(qualifier)

    statement = add_port_stats_reference(statement)
    return statement

if RUN_PORT_STATS:
  log.debug(f'Using file: {csv_path}')
  port_index = build_q62_port_index()
  deleted_ports = set()
  missing_ports = []
  updated_ports = set()
  rows_processed = 0

  with open(csv_path, 'r', encoding='utf-8', newline='') as csv_file:
      reader = csv.DictReader(csv_file, delimiter=';')

      for row in reader:
          puerto_base = row["\ufeffPUERTO BASE"].strip()
          year = row["AÑO"]
          num_barcos = row["NUM_BARCOS"]

          normalized_port = normalize_name(puerto_base)
          port_qid = port_index.get(normalized_port)

          if not port_qid:
              log.debug(f"No Q62 port found for '{puerto_base}'")
              missing_ports.append((puerto_base, year, num_barcos))
              continue

          try:
              if port_qid not in deleted_ports:
                  delete_existing_p45_statements(port_qid)
                  deleted_ports.add(port_qid)

              wd_item = loadEnID(port_qid)
              statement = build_p45_statement(num_barcos, year)

              wd_item.update(data=[statement])
              wd_item.write(login_instance, bot_account=True)

              updated_ports.add(port_qid)
              rows_processed += 1
              log.debug(f"{puerto_base} ({port_qid}) → year {year}, boats {num_barcos}")

          except Exception as e:
              log.error(f"Error updating {puerto_base} ({port_qid}) for year {year}: {e}")

  log.info("\n--- Port statistics summary ---")
  log.info(f"Rows processed: {rows_processed}")
  log.info(f"Ports updated: {len(updated_ports)}")
  log.info(f"Ports not found: {len(missing_ports)}")

  if missing_ports:
      log.debug("\nMissing ports:")
      for puerto_base, year, num_barcos in missing_ports:
          log.debug(f"  {puerto_base} | {year} | {num_barcos}")

# Vessels

## Vessel import configuration

This section defines how consolidated CSV columns map to Wikibase properties.

For each vessel field, the configuration specifies:
- target property ID.
- datatype.
- unit, when applicable.
- source origin (`eu`, `nat`, or both).

It also defines controlled mappings for hull materials, onboard equipment, and operating areas.

These mappings are central to reproducibility. If the Wikibase schema changes, this is the section that must be reviewed first.

In [ ]:
base_url = "https://effkg.wikibase.cloud/"
ref_prop = 'P46' #'Source'

In [ ]:
properties = {
  "CFR": { "property": "P12", "type": "string", "origin":["eu", "nat"] },
  "MSSI": { "property": "P15", "type": "string" , "origin":["eu"]},
  "Country of registration": { "property": "P55", "type": "string" , "origin":["eu", "nat"]},
  "Administration responsible of registration": { "property": "P56", "type": "string" , "origin":["nat"]},
  "Registration on national registry": { "property": "P31", "type": "time" , "origin":["nat"]},
  "GT Tonnage": { "property": "P20", "type": "quantity" , "unit": f"{base_url}entity/Q3" , "origin":["eu", "nat"]},
  "Other tonnage": { "property": "P21", "type": "quantity", "unit": f"{base_url}entity/Q3"  , "origin":["eu"]},
  "GTs": { "property": "P22", "type": "quantity", "unit": f"{base_url}entity/Q3"  , "origin":["eu"]},
  "Code": { "property": "P26", "type": "string" , "origin":["nat"]},
  "LOA": { "property": "P16", "type": "quantity", "unit": f"{base_url}entity/Q1" , "origin":["eu", "nat"]},
  "LBP": { "property": "P17", "type": "quantity", "unit": f"{base_url}entity/Q1"  , "origin":["eu"]},
  "Name": { "property": "P37", "type": "string" , "origin":["eu", "nat"]},
  "Registration number": { "property": "P10", "type": "string" , "origin":["eu", "nat"]},
  "External marking": { "property": "P9", "type": "string" , "origin":["eu", "nat"]},
  "Registration Place": { "property": "P11", "type": "wikibase-item" , "origin":["eu", "nat"]},
  "Status on national registry": { "property": "P32", "type": "string" , "origin":["nat"]},
  "Radio": { "property": "P28", "type": "wikibase-item" , "origin":["eu"]},
  "IRCS": { "property": "P28", "type": "wikibase-item" , "origin":["eu", "nat"]},
  "VMS": { "property": "P28", "type": "wikibase-item" , "origin":["eu"]},
  "ERS": { "property": "P28", "type": "wikibase-item" , "origin":["eu"]},
  "AIS": { "property": "P28", "type": "wikibase-item" , "origin":["eu"]},
  "UVI": { "property": "P14", "type": "string" , "origin":["eu"]},
  "IMO": { "property": "P49", "type": "string" , "origin":["nat"]},
  "Operates at": { "property": "P41", "type": "wikibase-item", "origin":["nat"] },
  "Fishing gear": { "property": "P25", "type": "wikibase-item" , "origin":["eu"]},
  "Main engine": { "property": "P23", "type": "quantity", "unit": [f"{base_url}entity/Q4", f"{base_url}entity/Q6"] , "origin":["eu", "nat"] },
  "Power of auxiliary engine": { "property": "P24", "type": "quantity", "unit": f"{base_url}entity/Q4", "origin":["eu"]},
  "Hull material": { "property": "P27", "type": "wikibase-item" , "origin":["eu", "nat"]},
  "Date of entry into service": { "property": "P29", "type": "time" , "origin":["eu"]},
  "Segment": { "property": "P59", "type": "string" , "origin":["eu"]},
  "Year of construction": { "property": "P30", "type": "time" , "origin":["eu", "nat"]},
  "Year of destruction": { "property": "P57", "type": "time" , "origin":["eu"]},
  "Year of retirement": { "property": "P58", "type": "time" , "origin":["eu"]}
}

materials = {
    "Acero": "Q7",
    "Metal": "Q397",
    "Aluminio": "Q547",
    "Fibra de Vidrio/Plástico": "Q548",
    "Madera": "Q549",
    "Madera forrada de Fibra": "Q550",
    "Poliester": "Q551",
    "Otros": "Q28603",
    "-": "Q28604"
}

materials_en = {
    "Steel": "Q7",
    "Metal": "Q397",
    "Aluminium": "Q547",
    "Fiber/Plastic": "Q548",
    "Wood": "Q549",
    "Wood coated with Fiber": "Q550",
    "Polyester": "Q551",
    "Other": "Q28603",
    "Unknown": "Q28604"
}

equipment = {
    "Radio": "Q8",
    "VMS": "Q9",
    "ERS": "Q10",
    "AIS": "Q11"
}

OPERATES_AT_MAP = {
    "GOLFO DE CADIZ": ["Q88"],
    "CANTABRICO NW": ["Q21"],
    "MEDITERRANEO": ["Q117"],
    "ZONAS CIEM VB, VI,VII y VIIIabde.": ["Q48", "Q49", "Q52", "Q79", "Q80", "Q81", "Q86"],
    "AGUAS DE PORTUGAL": ["Q87"],
    "VIIIabde.": ["Q79", "Q80", "Q81", "Q86"],
    "CANARIAS": ["Q100"]
}

## Checkpoint handling

The vessel import uses a JSON checkpoint file to track progress per input CSV file.

For each file, it stores the last processed row, last CFR, number of processed rows, created entities, updated entities, errors, and completion status.

This allows long imports to be resumed without starting from the beginning. Delete or edit the checkpoint only when you intentionally want to reprocess files.

In [ ]:
import json, os, time

CHECKPOINT_PATH = "./vessels_checkpoint.json"
CHECKPOINT_EVERY = 25

"""Load vessel import checkpoint data from disk."""
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"files": {}}

"""Persist checkpoint data atomically to disk."""
def save_checkpoint(state):
    tmp = CHECKPOINT_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CHECKPOINT_PATH)

"""Retrieve or initialize checkpoint metadata for a CSV file."""
def get_file_checkpoint(checkpoint, file_key):
    if file_key not in checkpoint["files"]:
        checkpoint["files"][file_key] = {
            "last_row": -1,
            "last_cfr": None,
            "processed": 0,
            "created": 0,
            "updated": 0,
            "errors": 0,
            "completed": False
        }
    return checkpoint["files"][file_key]

if RUN_VESSELS:
  checkpoint = load_checkpoint()

## Vessel import

This is the main import block.

For each consolidated vessel CSV file, the importer:
1. loads existing ports and vessels into lookup indexes.
2. reads each vessel row.
3. resolves whether the vessel already exists using CFR or external code.
4. builds labels and descriptions.
5. maps vessel type, category, dimensions, identifiers, equipment, fishing gear, dates, material, and registration place into Wikibase statements.
6. attaches references according to the source configuration.
7. creates or updates the Wikibase item.
8. updates the checkpoint.

The import is incremental and checkpoint-aware. If a file is marked as completed, it is skipped on later runs.

Dry run settings control whether the vessel import writes to Wikibase or only simulates changes.

When `DRY_RUN=True`, the notebook compares generated statements against existing statements and writes a report instead of modifying the graph.

Use dry-run mode before large imports or when validating new mappings.

In [ ]:
DRY_RUN = False
DRY_RUN_MAX_ROWS = None
DRY_RUN_START_ROW = 0
DRY_RUN_ONLY_NEW = False
DRY_RUN_REPORT_PATH = "./vessels_dry_run_report.json"
SAVE_CHECKPOINT_IN_DRY_RUN = False

if RUN_VESSELS:
  vessel_files = sorted(glob.glob(VESSELS_GLOB))
  log.debug(f'Using files: {vessel_files}')

  port_index = build_port_index()
  eu_ports_index = build_eu_ports_index()
  cnsp_ports_index = build_cnsp_ports_index()
  vessel_identity_index = build_vessel_identity_index()
  missing_ports = []

  for csv_path in vessel_files:
    file_key = os.path.basename(csv_path)
    file_cp = get_file_checkpoint(checkpoint, file_key)

    if file_cp.get("completed"):
        continue

    start_row = file_cp["last_row"] + 1

    with open(csv_path, 'r', encoding='utf-8', newline='') as csv_file:
        reader = csv.DictReader(csv_file, delimiter=';')
        properties_keys = list(properties.keys())
        for row_idx, row in enumerate(reader):
            if row_idx < start_row:
              continue
            if row_idx < DRY_RUN_START_ROW:
              continue

            if DRY_RUN and DRY_RUN_MAX_ROWS is not None:
                if row_idx >= DRY_RUN_START_ROW + DRY_RUN_MAX_ROWS:
                    log.info(f"Dry run stopped after {DRY_RUN_MAX_ROWS} rows.")
                    break
            cfr = row["CFR"].strip()
            identity_key = get_row_identity(row)
            existing_qid = vessel_identity_index.get(identity_key) if identity_key else None

            if DRY_RUN and DRY_RUN_ONLY_NEW and existing_qid is not None:
              continue
            if not is_missing_vessel_name(row["Name"]):
              item_label = row["Name"]
              description_en = f"Vessel named {item_label}, of CFR {cfr}"
              description_es = f"Embarcación denominada {item_label}, de CFR {cfr}"
            else:
              item_label = row["CFR"]
              description_en = f"Vessel of CFR {cfr}, with no name."
              description_es = f"Embarcación de CFR {cfr}, sin nombre."

            vessel_type = row["Vessel Type"]
            vessel_category_qid = get_vessel_category_qid(vessel_type)

            if vessel_category_qid == "Q466":
                instance_of_qid = "Q398"
                cat_ref = "nat"
            else:
                instance_of_qid = "Q24"
                cat_ref = ["eu", "nat"]

            statements = []

            # instance of
            statement = WI.wdi_core.WDItemID(value=instance_of_qid, prop_nr="P35")
            statement = add_new_reference(statement, "Vessel Type", origin_override=cat_ref)
            statements.append(statement)

            # has vessel category
            statement = WI.wdi_core.WDItemID(value=vessel_category_qid, prop_nr="P51")
            statement = add_new_reference(statement, "Vessel Type", origin_override=cat_ref)
            statements.append(statement)
            for column_name in properties_keys:
              value = row.get(column_name)
              if not value:
                  continue
              prop_id = properties[column_name]["property"]
              prop_type = properties[column_name]["type"]
              if prop_type == "string":
                if column_name == "Status on national registry": #Baja Definitiva (desde el 17/01/2000)
                  status_values = value.split(' ')
                  new_value = status_values[0] + ' ' + status_values[1]
                  statement = WI.wdi_core.WDString(value=new_value, prop_nr=prop_id)
                  date_parts = status_values[4].replace(')', '').split('/')
                  year = int(date_parts[2])
                  month = int(date_parts[1])
                  day = int(date_parts[0])
                  date = datetime(year, month, day, 0, 0, 0, 0)
                  iso_date = "+" + date.isoformat() + "Z"
                  qualifier = WI.wdi_core.WDTime(time=iso_date, prop_nr="P33", timezone=1, is_qualifier=True )#Since
                  statement.qualifiers.append(qualifier)
                  log.debug(f"  Added property {prop_id} with value '{new_value}', qualified by P33, qualifier value {iso_date}")
                  statement = add_new_reference(statement, column_name)
                  statements.append(statement)
                elif column_name == "CFR":
                  statement = WI.wdi_core.WDString(value=value, prop_nr=prop_id)
                  log.debug(f"  Added property {prop_id} with value '{value}'")
                  statement = add_new_reference(statement, column_name, cfr=value)
                  statements.append(statement)
                elif column_name == "Name" and is_missing_vessel_name(value):
                  continue
                elif column_name == "Operates at":
                  normalized_value = value.strip().upper()

                  if normalized_value not in OPERATES_AT_MAP:
                      log.error(f"Unknown 'Operates at' value '{value}' for CFR '{cfr}'")
                      continue

                  qids = OPERATES_AT_MAP[normalized_value]

                  for qid_value in qids:
                      statement = WI.wdi_core.WDItemID(
                          value=qid_value,
                          prop_nr="P41"
                      )

                      statement = add_new_reference(
                          statement,
                          column_name,
                          origin_override="nat"
                      )

                      statements.append(statement)
                else:
                  statement = WI.wdi_core.WDString(value=value, prop_nr=prop_id)
                  log.debug(f"  Added property {prop_id} with value '{value}'")
                  statement = add_new_reference(statement, column_name)
                  statements.append(statement)
              elif prop_type == "quantity":
                  unit = properties[column_name]["unit"]

                  if value == "0.00":
                    continue

                  if column_name == "Main engine":
                    engine_values = [v.strip() for v in value.split("|") if v.strip()]

                    for engine_value in engine_values:
                      if "kW" in engine_value and "CV" in engine_value:
                        # Example: 11,03 kW (15,0 CV)
                        parts = engine_value.split(" ")
                        kw_value = parts[0].replace(",", ".")
                        hp_value = parts[2].replace("(", "").replace(",", ".")

                        statement = WI.wdi_core.WDQuantity(
                            value=kw_value,
                            prop_nr=prop_id,
                            unit=unit[0]
                        )
                        log.debug(f"  Added property {prop_id} with RGFP kW value '{kw_value}'")
                        statement = add_new_reference(statement, column_name, origin_override="nat")
                        statements.append(statement)

                        statement = WI.wdi_core.WDQuantity(
                            value=hp_value,
                            prop_nr=prop_id,
                            unit=unit[1]
                        )
                        log.debug(f"  Added property {prop_id} with RGFP hp value '{hp_value}'")
                        statement = add_new_reference(statement, column_name, origin_override="nat")
                        statements.append(statement)

                      else:
                        # Simple european value in kw
                        new_value = engine_value.replace(",", ".")
                        statement = WI.wdi_core.WDQuantity(
                            value=new_value,
                            prop_nr=prop_id,
                            unit=unit[0]
                        )
                        log.debug(f"  Added property {prop_id} with EU kW value '{new_value}'")
                        statement = add_new_reference(statement, column_name, origin_override="eu")
                        statements.append(statement)

                  else:
                    statement = WI.wdi_core.WDQuantity(value=value, prop_nr=prop_id, unit=unit )
                    log.debug(f"  Added property {prop_id} with value '{value}'")
                    statement = add_new_reference(statement, column_name)
                    statements.append(statement)
              elif prop_type == "time":
                if '-' in value:
                    date_parts = value.split('-')
                    year = int(date_parts[0])
                    month = int(date_parts[1])
                    day = int(date_parts[2])
                else:
                    date_parts = value.split('/')
                    year = int(date_parts[2])
                    month = int(date_parts[1])
                    day = int(date_parts[0])
                iso_date = f"+{year:04d}-{month:02d}-{day:02d}T00:00:00Z"
                statement = WI.wdi_core.WDTime(time=iso_date, prop_nr=prop_id, timezone=1)
                log.debug(f"  Added property {prop_id} with value '{value}'")
                statement = add_new_reference(statement, column_name)
                statements.append(statement)
              elif prop_type == "wikibase-item":
                if column_name == "Registration Place":
                  port_id = None

                  for normalized_port in candidate_port_names(value):
                      port_id = port_index.get(normalized_port)
                      if port_id:
                          break

                  if not port_id:
                      port_id = find_or_create_port_from_csvs(value, eu_ports_index, cnsp_ports_index)

                  if not port_id:
                      log.debug(f"No port found for {value}")
                      missing_ports.append((item_label, value))
                      continue

                  statement = WI.wdi_core.WDItemID(value=port_id, prop_nr=prop_id)
                  log.debug(f"  Added property {prop_id} with value '{port_id}'")
                  statement = add_new_reference(statement, column_name)
                  statements.append(statement)
                elif column_name == "Hull material":
                  material_values = [v.strip() for v in value.split("|") if v.strip()]

                  for material_value in material_values:
                    if material_value in materials:
                      m_value = materials[material_value]
                      statement = WI.wdi_core.WDItemID(value=m_value, prop_nr=prop_id)
                      log.debug(f"  Added property {prop_id} with value '{m_value}'")
                      statement = add_new_reference(statement, column_name, origin_override="nat")
                      statements.append(statement)

                    elif material_value in materials_en:
                      m_value = materials_en[material_value]
                      statement = WI.wdi_core.WDItemID(value=m_value, prop_nr=prop_id)
                      log.debug(f"  Added property {prop_id} with value '{m_value}'")
                      statement = add_new_reference(statement, column_name, origin_override="eu")
                      statements.append(statement)

                    else:
                      log.error(f"Unknown hull material '{material_value}' for vessel '{item_label}'")
                      continue
                elif column_name in ["Radio", "VMS", "ERS", "AIS"]:
                  if value == "Y":
                    new_value = equipment[column_name]
                    statement = WI.wdi_core.WDItemID(value=new_value, prop_nr=prop_id)
                    log.debug(f"  Added property {prop_id} with value '{new_value}'")
                    if column_name == "Radio" and row["IRCS"] != "":
                      qualifier = WI.wdi_core.WDString(value=row["IRCS"], prop_nr="P13", is_qualifier=True )
                      statement.qualifiers.append(qualifier)
                      log.debug(f"  ...with qualifier P13 (IRCS) '{row["IRCS"]}'")
                    statement = add_new_reference(statement, column_name)
                    statements.append(statement)
                elif column_name == "Fishing gear":
                  fg_values = row[column_name].split(", ")
                  for fg_value in fg_values:
                    fg_qid = get_fishing_gear_qid(fg_value)
                    statement = WI.wdi_core.WDItemID(value=fg_qid, prop_nr=prop_id)
                    log.debug(f"  Added property {prop_id} with value '{fg_qid}'")
                    statement = add_new_reference(statement, column_name)
                    statements.append(statement)

            try:
                if existing_qid:
                    new_item = WI.wdi_core.WDItemEngine(
                        wd_item_id=existing_qid,
                        data=statements,
                        mediawiki_api_url=mediawiki_api_url
                    )
                else:
                    new_item = WI.wdi_core.WDItemEngine(
                        data=statements,
                        mediawiki_api_url=mediawiki_api_url
                    )

                new_item.set_label(item_label, lang='en')
                new_item.set_label(item_label, lang='es')
                new_item.set_description(description_en, lang='en')
                new_item.set_description(description_es, lang='es')

                dry_run_stats["processed"] += 1

                if DRY_RUN:
                    if existing_qid:
                        current_grouped = load_existing_statement_signatures(existing_qid)
                        new_grouped = group_statement_signatures(statements)

                        changed_props = []
                        for prop, new_values in new_grouped.items():
                            old_values = current_grouped.get(prop, set())
                            if old_values != new_values:
                                changed_props.append({
                                    "property": prop,
                                    "old": sorted(list(old_values)),
                                    "new": sorted(list(new_values))
                                })

                        if changed_props:
                            dry_run_stats["would_update"] += 1
                            dry_run_stats["details"].append({
                                "qid": existing_qid,
                                "identity_key": identity_key,
                                "label": item_label,
                                "action": "update",
                                "changed_properties": changed_props
                            })
                        else:
                            dry_run_stats["would_skip_unchanged"] += 1
                    else:
                        dry_run_stats["would_create"] += 1
                        dry_run_stats["details"].append({
                            "qid": None,
                            "identity_key": identity_key,
                            "label": item_label,
                            "action": "create"
                        })

                    result = existing_qid or "DRY_RUN"
                else:
                    result = new_item.write(login_instance, bot_account=True)

                    if not existing_qid and result and identity_key:
                        vessel_identity_index[identity_key] = result

                    if existing_qid:
                        file_cp["updated"] += 1
                    else:
                        file_cp["created"] += 1

                file_cp["last_row"] = row_idx
                file_cp["last_cfr"] = cfr
                file_cp["processed"] += 1

                log.debug(f"Processed {item_label} with QID {result}")

            except Exception as e:
                file_cp["last_row"] = row_idx
                file_cp["last_cfr"] = cfr
                file_cp["errors"] += 1
                dry_run_stats["errors"] += 1
                log.error(f"Error at row {row_idx}, CFR {cfr}: {e}")

            if row_idx % CHECKPOINT_EVERY == 0:
              if (not DRY_RUN) or SAVE_CHECKPOINT_IN_DRY_RUN:
                  save_checkpoint(checkpoint)

    log.debug("\n--- Missing ports report ---")

    for vessel, port in missing_ports:
        log.debug(f"{vessel} → {port}")

    log.debug(f"\nTotal missing ports: {len(missing_ports)}")
    file_cp["completed"] = True
    if (not DRY_RUN) or SAVE_CHECKPOINT_IN_DRY_RUN:
        save_checkpoint(checkpoint)
    start_time = time.time()

    if row_idx % 50 == 0:
        elapsed = time.time() - start_time
        rate = row_idx / elapsed if elapsed > 0 else 0
        log.info(f"{row_idx} vessels | {rate:.2f} rows/sec")

if DRY_RUN:
  with open(DRY_RUN_REPORT_PATH, "w", encoding="utf-8") as f:
      json.dump(dry_run_stats, f, ensure_ascii=False, indent=2)

  log.info("\n--- DRY RUN SUMMARY ---")
  log.info(f"Processed: {dry_run_stats['processed']}")
  log.info(f"Would create: {dry_run_stats['would_create']}")
  log.info(f"Would update: {dry_run_stats['would_update']}")
  log.info(f"Would skip unchanged: {dry_run_stats['would_skip_unchanged']}")
  log.info(f"Errors: {dry_run_stats['errors']}")
  log.info(f"Report: {DRY_RUN_REPORT_PATH}")